In [1]:
import pandas as pd

# Load data
df = pd.read_csv("merged_output_all.csv", dtype=str)
print("The dataframe size of the merged output:", df.size)

# Convert DATE_RECEIVED to datetime
df['DATE_RECEIVED'] = pd.to_datetime(df['DATE_RECEIVED'], errors='coerce')

# Filter for dental implants only
df = df[df['GENERIC_NAME'] == 'DENTAL IMPLANT']

# Drop rows with missing DATE_RECEIVED or FOI_TEXT
df = df.dropna(subset=['DATE_RECEIVED', 'FOI_TEXT'])

# Extract year and month for trend analysis (optional)
df['year'] = df['DATE_RECEIVED'].dt.year

# Keep only the specified columns
df = df[["MDR_REPORT_KEY", "MANUFACTURER_D_NAME", "FOI_TEXT", "year"]]

# Preview
df.head()

The dataframe size of the merged output: 138115598


,MDR_REPORT_KEY,MANUFACTURER_D_NAME,FOI_TEXT,year
0,9537111,IMPLANT DIRECT SYBRON MANUFACTURING LLC,"PER COMPLAINT (B)(4), AFTER CLINICAL PROCEDURE...",2020
1,9537306,THOMMEN MEDICAL AG,"LOSS OF OSSEOINTEGRATION, PRIMARY STABILITY WA...",2020
2,9537307,THOMMEN MEDICAL AG,"IMPLANT DIDN'T ACHIEVE OSSEOINTEGRATION, IMPLA...",2020
3,9537308,THOMMEN MEDICAL AG,"IMPLANT DIDN'T ACHIEVE OSSEOINTEGRATION, PRIMA...",2020
4,9537309,THOMMEN MEDICAL AG,"IMPLANT DIDN'T ACHIEVE OSSEOINTEGRATION, PRIMA...",2020


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 260103 entries, 0 to 3634620
Data columns (total 4 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   MDR_REPORT_KEY       260103 non-null  object
 1   MANUFACTURER_D_NAME  259986 non-null  object
 2   FOI_TEXT             260103 non-null  object
 3   year                 260103 non-null  int32 
dtypes: int32(1), object(3)
memory usage: 8.9+ MB


In [3]:
print("The dataframe size of the filtered output for DENTAL IMPLANT:", df.size)

The dataframe size of the filtered output for DENTAL IMPLANT: 1040412


Data Subset Prepare

In [4]:
# Total sample size
N = 10000

# Compute sample size per year
year_counts = df["year"].value_counts().sort_index()
year_ratios = year_counts / year_counts.sum()
year_samples = (year_ratios * N).round().astype(int)

print(year_samples)

# Stratified sampling
df_sample = (
    df.groupby("year", group_keys=False)
      .apply(lambda x: x.sample(n=year_samples.loc[x.name], random_state=42), include_groups=True)
      .reset_index(drop=True)
)

#  'year' column is already in df_subset — no need to re-derive it
print("\n Sampled subset size by year:")
print(df_sample["year"].value_counts().sort_index())


year
2020    1608
2021    2088
2022    1934
2023    1737
2024    2632
Name: count, dtype: int64

 Sampled subset size by year:
year
2020    1608
2021    2088
2022    1934
2023    1737
2024    2632
Name: count, dtype: int64


/var/folders/_6/bkb79v1n1qgb5pjgnzbz79nw0000gn/T/ipykernel_19209/1734446655.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=year_samples.loc[x.name], random_state=42), include_groups=True)


In [5]:
print(df_sample.columns)

Index(['MDR_REPORT_KEY', 'MANUFACTURER_D_NAME', 'FOI_TEXT', 'year'], dtype='object')


## RAG with PDFs

Load & Chunk PDF Texts

In [6]:
#!pip install -U langchain-community
#!pip install pypdf
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load all PDFs
pdf_paths = [
    "RAG text/a-dentists-guide-to-implantology.pdf",
    "RAG text/implant_book.pdf",
    "RAG text/Surgical-Implant-Manual-web.pdf"
]

all_chunks = []
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

for path in pdf_paths:
    loader = PyPDFLoader(path)
    pages = loader.load()
    chunks = splitter.split_documents(pages)
    all_chunks.extend(chunks)


Create Embeddings & Vector Store

In [7]:
#!pip install -U langchain-huggingface
#!pip install tqdm
#!pip install sentence-transformers
#!pip install faiss-cpu
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from tqdm import tqdm

# Step 1: Load embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

texts = [doc.page_content for doc in all_chunks]
metadatas = [doc.metadata for doc in all_chunks]

# Step 2: Generate embeddings with progress
print("🔁 Generating embeddings...")
embeddings = []
for text in tqdm(texts, desc="Embedding Chunks"):
    vector = embedding_model.embed_query(text)
    embeddings.append(vector)

# Step 3: Zip into (text, embedding) pairs
text_embedding_pairs = list(zip(texts, embeddings))

# Step 4: Build the FAISS vectorstore
print("📦 Building FAISS vector store...")
vectorstore = FAISS.from_embeddings(
    text_embedding_pairs,
    embedding_model,
    metadatas=metadatas
)

/var/folders/_6/bkb79v1n1qgb5pjgnzbz79nw0000gn/T/ipykernel_19209/2803919526.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


🔁 Generating embeddings...


Embedding Chunks: 100%|██████████| 694/694 [00:16<00:00, 43.19it/s]


📦 Building FAISS vector store...


In [8]:
vectorstore.save_local("rag_vectorstore")

Build the Retriever & LLM Chain

In [138]:
#!pip install -U langchain_ollama
from langchain.chains import RetrievalQA
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.2")

retriever = vectorstore.as_retriever()

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)


In [31]:
# Install the updated package first
!pip install -U langchain-openai

# Then use:
from langchain_openai import ChatOpenAI

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.8/767.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 988.7/988.7 kB 4.8 MB/s eta 0:00:00a 0:00:01


In [114]:
from langchain.chat_models import ChatOpenAI
openai_api_key="YOUR OPENAI API KEY"
llm_online = ChatOpenAI(
    model="gpt-3.5-turbo",
    openai_api_key=openai_api_key
)

rag_chain_online = RetrievalQA.from_chain_type(
    llm=llm_online,
    retriever=retriever,
    return_source_documents=True
)

In [115]:
query = "What are common reasons for implant failure?"
#result = rag_chain(query)
result = rag_chain_online.invoke({"query": query})

print("Answer:\n", result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(doc.metadata.get("source", "unknown"))

Answer:
 Common reasons for dental implant failure include poor surgical technique, inability to achieve primary fixation, inadvertent implant loading during the integration phase, infection, systemic conditions like uncontrolled diabetes, plaque-induced peri-implant disease, unfavourable loading conditions due to poor restorative design, and failure to control occlusal interferences.

Sources:
RAG text/a-dentists-guide-to-implantology.pdf
RAG text/a-dentists-guide-to-implantology.pdf
RAG text/a-dentists-guide-to-implantology.pdf
RAG text/a-dentists-guide-to-implantology.pdf
RAG text/a-dentists-guide-to-implantology.pdf


Ask Questions Using Your PDF Knowledge

In [16]:
query = "What are common reasons for implant failure?"
#result = rag_chain(query)
result = rag_chain.invoke({"query": query})

print("Answer:\n", result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(doc.metadata.get("source", "unknown"))


Answer:
 According to the context provided, some common reasons for early implant failures include:

1. Poor surgical technique
2. Inability to achieve primary fixation
3. Inadvertent implant loading during the integration phase
4. Infection
5. Systemic conditions such as uncontrolled diabetes.

Late implant failures can be identified and treated successfully if they are intercepted in the early stages of the disease process, but it's not specified what common reasons there are for late implant failures.

Sources:
RAG text/a-dentists-guide-to-implantology.pdf
RAG text/a-dentists-guide-to-implantology.pdf
RAG text/a-dentists-guide-to-implantology.pdf
RAG text/a-dentists-guide-to-implantology.pdf


In [14]:
FAISS.load_local("rag_vectorstore", embedding_model, allow_dangerous_deserialization=True)

In [29]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM
from langchain.chains import RetrievalQA
import pandas as pd

# Load dataset
eval_df = pd.read_csv("rag_eval_dataset_expanded.csv")

# Load vectorstore
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.load_local("rag_vectorstore", embedding_model, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})  # k = 5

# Initialize metrics
hit_count = 0
reciprocal_ranks = []
precision_scores = []
recall_scores = []

# Evaluation loop
for idx, row in eval_df.iterrows():
    query = row["query"]
    expected_source = row["source"].lower().split(",")[0].strip()  # Just the filename

    retrieved_docs = retriever.get_relevant_documents(query)
    matches = [doc for doc in retrieved_docs if expected_source in doc.metadata.get("source", "").lower()]
    match_ranks = [i for i, doc in enumerate(retrieved_docs) if expected_source in doc.metadata.get("source", "").lower()]

    # Metrics
    hit_at_k = 1 if matches else 0
    hit_count += hit_at_k
    rr = 1 / (match_ranks[0] + 1) if match_ranks else 0
    reciprocal_ranks.append(rr)
    precision = len(matches) / len(retrieved_docs)
    recall = 1 if matches else 0
    precision_scores.append(precision)
    recall_scores.append(recall)

    # 🔍 Generate answer with RAG chain
    result = rag_chain.invoke({"query": query})
    generated_answer = result["result"]

    print("="*80)
    print(f"Query {idx+1}: {query}")
    print(f"Expected Source: {expected_source}")
    print(f"Retrieved Sources:")
    for i, doc in enumerate(retrieved_docs):
        print(f"  {i+1}. {doc.metadata.get('source', 'unknown')}")
    print(f"\n🔹 Generated Answer:\n{generated_answer}\n")
    print(f"✅ Match Found: {bool(matches)} | 🎯 Rank: {match_ranks[0] + 1 if match_ranks else 'N/A'}")
    print("="*80 + "\n")

# Aggregate summary
total_queries = len(eval_df)
print("=== FINAL EVALUATION ===")
print(f"Total Queries: {total_queries}")
print(f"Hit@5: {hit_count / total_queries:.3f}")
print(f"MRR: {sum(reciprocal_ranks) / total_queries:.3f}")
print(f"Precision@5: {sum(precision_scores) / total_queries:.3f}")
print(f"Recall@5: {sum(recall_scores) / total_queries:.3f}")

Query 1: What are the indications for zygomatic implants?
Expected Source: surgical-implant-manual-web.pdf
Retrieved Sources:
  1. RAG text/implant_book.pdf
  2. RAG text/a-dentists-guide-to-implantology.pdf
  3. RAG text/Surgical-Implant-Manual-web.pdf
  4. RAG text/implant_book.pdf
  5. RAG text/Surgical-Implant-Manual-web.pdf

🔹 Generated Answer:
I don't know the answer to what are the indications for zygomatic implants based on the provided context.

✅ Match Found: True | 🎯 Rank: 3

Query 2: What is the difference between subperiosteal and endosteal implants?
Expected Source: implant_book.pdf
Retrieved Sources:
  1. RAG text/a-dentists-guide-to-implantology.pdf
  2. RAG text/a-dentists-guide-to-implantology.pdf
  3. RAG text/a-dentists-guide-to-implantology.pdf
  4. RAG text/a-dentists-guide-to-implantology.pdf
  5. RAG text/implant_book.pdf

🔹 Generated Answer:
According to the provided context, the main difference between subperiosteal and endosteal implants lies in their placeme

In [169]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
# Optional generation (set DO_GENERATION=True and configure LLM if you want answers)
from langchain_ollama import OllamaLLM
from langchain.chains import RetrievalQA

import pandas as pd

# --------------------
# Config
# --------------------
K_LIST = [1, 2, 4, 8, 16, 50]   # k values to compare
DATA_CSV = "rag_eval_dataset_expanded.csv"
VSTORE_DIR = "rag_vectorstore"
EMB_MODEL = "all-MiniLM-L6-v2"
DO_GENERATION = False            # set True if you want to run RAG answers per k

# --------------------
# Load data and vector store
# --------------------
eval_df = pd.read_csv(DATA_CSV)

embedding_model = HuggingFaceEmbeddings(model_name=EMB_MODEL)
vectorstore = FAISS.load_local(
    VSTORE_DIR, embedding_model, allow_dangerous_deserialization=True
)

# --------------------
# Metric containers
# --------------------
summary_rows = []
per_query_rows = []   # optional detailed rows per query & k

for k in K_LIST:
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    hit_count = 0
    reciprocal_ranks = []
    precision_scores = []
    recall_scores = []

    # Optional: build a chain tied to this retriever (only if you need generations)
    if DO_GENERATION:
        llm = OllamaLLM(model="llama3.2")  # change to your local model tag
        rag_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

    for idx, row in eval_df.iterrows():
        query = row["query"]
        # assume a single ground-truth source filename stored in 'source'
        expected_source = row["source"].lower().split(",")[0].strip()

        retrieved_docs = retriever.get_relevant_documents(query)

        # ranks where expected source appears
        match_ranks = [
            i for i, doc in enumerate(retrieved_docs)
            if expected_source in str(doc.metadata.get("source", "")).lower()
        ]
        matches = len(match_ranks)

        # ----- metrics @k -----
        hit_at_k = 1 if matches > 0 else 0
        hit_count += hit_at_k

        rr = 1 / (match_ranks[0] + 1) if match_ranks else 0.0
        reciprocal_ranks.append(rr)

        precision = matches / len(retrieved_docs) if retrieved_docs else 0.0
        precision_scores.append(precision)

        # with a single relevant item per query, Recall@k == Hit@k
        recall = 1.0 if matches > 0 else 0.0
        recall_scores.append(recall)

        # ----- (Optional) generate an answer -----
        if DO_GENERATION:
            result = rag_chain.invoke({"query": query})
            generated_answer = result["result"]
        else:
            generated_answer = None

        # save detailed row
        per_query_rows.append({
            "k": k,
            "query_id": idx + 1,
            "query": query,
            "expected_source": expected_source,
            "topk_sources": [doc.metadata.get("source", "unknown") for doc in retrieved_docs],
            "hit@k": hit_at_k,
            "rank": (match_ranks[0] + 1) if match_ranks else None,
            "rr": rr,
            "precision@k": precision,
            "recall@k": recall,
            "answer": generated_answer
        })

    # aggregate per-k
    total_q = len(eval_df)
    summary_rows.append({
        "k": k,
        "Hit@k": hit_count / total_q,
        "MRR@k": sum(reciprocal_ranks) / total_q,
        "Precision@k": sum(precision_scores) / total_q,
        "Recall@k": sum(recall_scores) / total_q,  # same as Hit@k under single-label assumption
        "Queries": total_q
    })

# --------------------
# Print summaries
# --------------------
summary_df = pd.DataFrame(summary_rows).sort_values("k")
print("\n=== RETRIEVAL EVALUATION BY k ===")
print(summary_df.to_string(index=False))

# If you want the detailed per-query log as a CSV:
# pd.DataFrame(per_query_rows).to_csv("retrieval_eval_per_query.csv", index=False)


=== RETRIEVAL EVALUATION BY k ===
 k    Hit@k    MRR@k  Precision@k  Recall@k  Queries
 1 0.428571 0.428571     0.428571  0.428571       21
 2 0.523810 0.476190     0.428571  0.523810       21
 4 0.666667 0.519841     0.380952  0.666667       21
 8 0.809524 0.545238     0.416667  0.809524       21
16 0.904762 0.553930     0.428571  0.904762       21
50 1.000000 0.557599     0.390476  1.000000       21


In [168]:
# --- set retriever for production use (k = 16) ---
prod_retriever = vectorstore.as_retriever(search_kwargs={"k": 16})

# (optional) build your RAG chain with this retriever
llm = OllamaLLM(model="llama3.2")  # or your chosen model
rag_chain = RetrievalQA.from_chain_type(llm=llm, retriever=prod_retriever)

# example query
query = "What are common causes of peri-implantitis?"
result = rag_chain.invoke({"query": query})
print(result["result"])

According to the text, two common causes of peri-implantitis are:

1. Biological failures: caused by plaque-induced peri-implant disease.
2. Mechanical failures: caused by unfavourable loading conditions due to poor restorative design or failure to control occlusal interferences.

Additionally, other factors that may contribute to the development of peri-implantitis include smoking, genetic factors, and poor oral hygiene.


## Dataset Prepare for Queries  

Extract causes

In [170]:
def extract_cause_rag(text):
    try:
        query_prompt = f"""
You are a medical analyst reviewing dental implant adverse event reports.

Your task is to read the report below and analyse why the adverse event happened, determine if **any of the following causes are clearly stated**:
[SMOKING, OSSEOINTEGRATION, PAIN, INFLAMMATION, INFECTION].

Instructions:
- Only choose a cause **if it is explicitly mentioned or clearly described** in the report.
- Return **only one cause** — the one most clearly indicated.
- If no clear cause is stated, return "NO CLEAR CAUSE".

Report:
{text}

Answer with only one of the following options:
SMOKING, OSSEOINTEGRATION, PAIN, INFLAMMATION, INFECTION, NO CLEAR CAUSE
""".strip()

        response = rag_chain.invoke({"query": query_prompt})["result"].strip().upper()

        for cause in valid_causes:
            if cause in response:
                return cause
        return "NO CLEAR CAUSE"
    except Exception as e:
        print("Error:", e)
        return "NO CLEAR CAUSE"


In [171]:
valid_causes = ["PST", "IPF", "IIL", "INF", "UCD", "BF", "MF", "OSS", "NO CLEAR CAUSE"]

def extract_cause_rag(text):
    try:
        query_prompt = f"""
You are a medical analyst reviewing dental implant adverse event reports.

Your task is to read the report below and analyze why the adverse event happened. Determine if **any of the following causes are clearly stated**:

[ PST, IPF, IIL, INF, UCD, BF, MF, OSS ]

Definitions:
- PST: Poor surgical technique
- IPF: Inability to achieve primary fixation
- IIL: Inadvertent implant loading during integration phase
- INF: Infection
- UCD: Uncontrolled diabetes
- BF: Biological failure (e.g., plaque-induced peri-implant disease)
- MF: Mechanical failure (e.g., unfavorable loading, poor restorative design)
- OSS: Osseointegration failure

Instructions:
- Choose **only one cause**, and only if it is explicitly mentioned or clearly described in the report.
- If no clear cause is described, return "NO CLEAR CAUSE".
- Respond with **exactly one** of: PST, IPF, IIL, INF, UCD, BF, MF, OSS, NO CLEAR CAUSE

Report:
{text}

Respond with one of the above codes only.
""".strip()

        response = rag_chain.invoke({"query": query_prompt})["result"].strip().upper()

        # Return exact match if found
        for cause in valid_causes:
            if cause == response:
                return cause

        # If a valid cause was mentioned as part of a longer output
        for cause in valid_causes:
            if cause in response:
                return cause

        return "NO CLEAR CAUSE"
    except Exception as e:
        print("Error:", e)
        return "NO CLEAR CAUSE"


labeling the data with cuases 

In [46]:
labeled_causes = []
total = len(df_sample)

print(f"Starting cause classification using RAG for {total} records...\n")

for i, text in enumerate(df_sample["FOI_TEXT"]):
    labeled_causes.append(extract_cause_rag(text))
    if (i + 1) % 100 == 0 or (i + 1) == total:
        print(f"Processed {i + 1} / {total}")


Starting cause classification using RAG for 9999 records...

Processed 100 / 9999
Processed 200 / 9999
Processed 300 / 9999
Processed 400 / 9999
Processed 500 / 9999
Processed 600 / 9999
Processed 700 / 9999
Processed 800 / 9999
Processed 900 / 9999
Processed 1000 / 9999
Processed 1100 / 9999
Processed 1200 / 9999
Processed 1300 / 9999
Processed 1400 / 9999
Processed 1500 / 9999
Processed 1600 / 9999
Processed 1700 / 9999
Processed 1800 / 9999
Processed 1900 / 9999
Processed 2000 / 9999
Processed 2100 / 9999
Processed 2200 / 9999
Processed 2300 / 9999
Processed 2400 / 9999
Processed 2500 / 9999
Processed 2600 / 9999
Processed 2700 / 9999
Processed 2800 / 9999
Processed 2900 / 9999
Processed 3000 / 9999
Processed 3100 / 9999
Processed 3200 / 9999
Processed 3300 / 9999
Processed 3400 / 9999
Processed 3500 / 9999
Processed 3600 / 9999
Processed 3700 / 9999
Processed 3800 / 9999
Processed 3900 / 9999
Processed 4000 / 9999
Processed 4100 / 9999
Processed 4200 / 9999
Processed 4300 / 9999
Pr

 Save results

In [47]:
df_sample["cause"] = labeled_causes
df_sample.to_csv("labeled_dental_reports_rag.csv", index=False)
print("\n RAG-based labeling complete. Saved as 'labeled_dental_reports_rag.csv'")



 RAG-based labeling complete. Saved as 'labeled_dental_reports_rag.csv'


In [49]:
query = """What is the influence of infection on dental implant failure?"""

# Ask the retriever directly
result = rag_chain(query)

print("Answer:\n", result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(doc.metadata.get("source", "unknown"))


Answer:
 According to the text, infections can cause dental implant failure due to chronic, localized infections that evade the host's defense mechanisms and can lead to excessive movement of the implant, resulting in failure. Chronic infections may also be a reason for not loading or even placing the implant. However, it is noted that infections are typically localized to a specific site rather than becoming systemic.

Sources:
RAG text/a-dentists-guide-to-implantology.pdf
RAG text/implant_book.pdf
RAG text/implant_book.pdf
RAG text/a-dentists-guide-to-implantology.pdf


In [62]:
import pandas as pd
# Load the full dataset
df = pd.read_csv("labeled_dental_reports_rag.csv")
# Extract first 100 rows and only the specified columns
subset_df = df.loc[:99, ["FOI_TEXT", "cause"]]
# Optionally: Save to a new CSV file
subset_df.to_csv("subset_labeled_dental_reports.csv", index=False)
# Display the first few rows
print(subset_df.head())


                                            FOI_TEXT           cause
0  PER COMPLAINT (B)(4), DURING CLINICAL PROCEDUR...             OSS
1  FOLLOW-UP SUBMITTED TO REPORT DEVICE EVALUATIO...  NO CLEAR CAUSE
2  FOLLOW-UP SUBMITTED TO REPORT DEVICE EVALUATIO...  NO CLEAR CAUSE
3  INCLUDED NI FOR SECTION TO INDICATE NO INFORMA...  NO CLEAR CAUSE
4  THIS REPORT IS BEING SUBMITTED TO RELAY ADDITI...  NO CLEAR CAUSE


In [172]:
def generate_sql_prompt(question):
    return f"""
You are a helpful data assistant. Given the user question below:

{question}

Generate a SQL query to run on the table `labeled_dental_reports`.

This table has the following columns:
- MANUFACTURER_D_NAME: string — name of the manufacturer
- cause: string — categorical value indicating the reported issue
- year: integer — the report year (e.g., 2020 to 2024)

Valid values for the `cause` column:
- PST: Poor surgical technique
- IPF: Inability to achieve primary fixation
- IIL: Inadvertent implant loading during integration phase
- INF: Infection
- UCD: Uncontrolled diabetes
- BF: Biological failures
- MF: Mechanical failures
- OSS: Osseointegration failure
- NO CLEAR CAUSE: No specific cause mentioned

Instructions:
- Use **only** the above columns — do not generate or reference any others.
- Always align your SQL structure with the user intent.
- Use the following rules when forming queries:
  • For **trends over time** → use `GROUP BY year ORDER BY year`
  • For **rankings** → use `ORDER BY ... DESC/ASC LIMIT ...`
  • For **comparisons** → consider use of `OFFSET`, `HAVING`, or `DISTINCT`
  • For **year-specific queries** → filter with `WHERE year = ...`
  • For **specific failure types** → filter with `WHERE cause = '...'`
  • For **manufacturer analysis** → use `GROUP BY MANUFACTURER_D_NAME`
  • For **combined year filters** → use `WHERE year IN (...)` or `BETWEEN ...`
  • For **multiple fields** (e.g. year and cause) → use `GROUP BY field1, field2`
- Avoid aggregating or selecting columns not grouped or explicitly aggregated.
- Do not use undefined aliases or invalid functions.
- Do not add conditions or filters unless clearly asked in the question.
- Do not exclude rows with 'NO CLEAR CAUSE' unless explicitly mentioned.
  - If the question specifies a particular year (e.g., "in 2024"), include a `WHERE year = ...` clause. Do not generalize to multiple years or use `GROUP BY year` unless explicitly asked.
- Do not include explanatory text, comments, or markdown formatting.
- Use `snake_case` for aliases and **uppercase SQL keywords**.
- Your SQL should follow this structure:

  SELECT [columns]
  FROM labeled_dental_reports
  [WHERE ...]
  [GROUP BY ...]
  [HAVING ...]
  [ORDER BY ...]
  [LIMIT ...]

Examples:

What are the most common causes of implant failure?
Generated SQL:
SELECT
    cause,
    COUNT(*) as count
FROM labeled_dental_reports
GROUP BY cause
ORDER BY count DESC;

Which year had the lowest number of implant failures?
Generated SQL:
SELECT
    year,
    COUNT(*) as count 
FROM labeled_dental_reports
GROUP BY year
ORDER BY count ASC
LIMIT 1;

Show the yearly trend of infection cases.
Generated SQL:
SELECT
    year,
    COUNT(*) as infection_cases
FROM labeled_dental_reports
WHERE cause = 'INF'
GROUP BY year
ORDER BY year;

How many cases were reported in 2023?
Generated SQL:
SELECT
    COUNT(*) as case_count
FROM labeled_dental_reports
WHERE year = 2023;

Which manufacturer had the most osseointegration failures in 2021?
Generated SQL:
SELECT
    MANUFACTURER_D_NAME,
    COUNT(*) as count
FROM labeled_dental_reports
WHERE year = 2021 AND cause = 'OSS'
GROUP BY MANUFACTURER_D_NAME
ORDER BY count DESC
LIMIT 1;

Which cause had the second highest number of reports?
Generated SQL:
SELECT
    cause,
    COUNT(*) as count
FROM labeled_dental_reports
GROUP BY cause
ORDER BY count DESC
LIMIT 1 OFFSET 1;
""".strip()


In [173]:
def get_clean_sql(question, llm):
    prompt = generate_sql_prompt(question)
    raw_response = llm.invoke(prompt).strip()

    # Split and select only the first valid SQL block
    if "```sql" in raw_response:
        sql_code = raw_response.split("```sql")[1].split("```")[0].strip()
    else:
        sql_code = raw_response.strip()

    # 🛑 Truncate at first semicolon if multiple queries exist
    if ";" in sql_code:
        sql_code = sql_code.split(";")[0].strip()

    return sql_code

In [174]:
import pandasql as ps

def run_sql(sql, df_context):
    try:
        result_df = ps.sqldf(sql, df_context)
        return result_df
    except Exception as e:
        print("⚠️ SQL execution failed:", e)
        return None

In [175]:
def generate_answer_from_result(question, result_df, llm=None, sql=None, include_sql=False):
    if result_df is None or result_df.empty:
        return "Sorry, I couldn’t find any relevant data to answer your question."

    # Mapping cause codes to human-readable labels
    cause_labels = {
        "INF": "Infection",
        "BF": "Biological failure",
        "MF": "Mechanical failure",
        "OSS": "Osseointegration failure",
        "UCD": "Uncontrolled diabetes",
        "PST": "Poor surgical technique",
        "IPF": "Inability to achieve primary fixation",
        "IIL": "Inadvertent implant loading",
        "NO CLEAR CAUSE": "No clear cause",
    }

    # Replace cause codes in the result DataFrame for better readability
    if "cause" in result_df.columns:
        result_df["cause"] = result_df["cause"].apply(lambda c: cause_labels.get(c, c))

    try:
        top_row = result_df.sort_values(by=result_df.columns[-1], ascending=False).iloc[0]
    except:
        top_row = result_df.iloc[0] if not result_df.empty else None

    question_upper = question.upper()

    for code, description in cause_labels.items():
        if code in question_upper or description.upper() in question_upper:
            if "MANUFACTURER" in question_upper:
                manufacturer = top_row.iloc[0]
                count = top_row.iloc[1] if len(top_row) > 1 else "unknown"
                return f"{result_df.to_string(index=False)}\n\nThe manufacturer with the highest number of {description.lower()} cases is {manufacturer} with {count} reports."

    if "cause" in result_df.columns and result_df.shape[1] == 1:
        cause = top_row.iloc[0]
        return f"{result_df.to_string(index=False)}\n\nThe most frequently reported cause is {cause}."

    if llm:
        table_text = result_df.to_string(index=False)
        prompt = f"""You are a helpful data analyst assistant.

Given the user question:
"{question}"

And the following table result:
{table_text}

Write a clear and concise human-readable answer. Be factual and helpful.
"""
        response = llm.invoke(prompt).strip()
        return f"{table_text}\n\n{response}" if not include_sql else f"{table_text}\n\n{response}\n\n(SQL: {sql})"

    return result_df.to_string(index=False)

In [176]:
df_sample = pd.read_csv("labeled_dental_reports_rag.csv")

In [177]:
def is_sql_question(question, llm):
    routing_prompt = f"""
You are a smart assistant helping users with dental implant questions.

You can access a table with the following columns:
- MANUFACTURER_D_NAME (manufacturer name)
- cause (categorical reason for failure)
- year (report year)

Valid values for `cause` include:
- PST: Poor surgical technique
- IPF: Inability to achieve primary fixation
- IIL: Inadvertent implant loading
- INF: Infection
- UCD: Uncontrolled diabetes
- BF: Biological failures
- MF: Mechanical failures
- OSS: Osseointegration failure
- NO CLEAR CAUSE: Unclear or unspecified

A user asks:
"{question}"

If the question requires querying counts, comparisons, or trends from the table — such as:
- What are the most common causes of implant failure?
- Which manufacturer had more infections?
- Show the yearly trend of implant failure.

Respond with: SQL

If the question asks for general knowledge (e.g., definitions, effects) that cannot be answered from the table:
- What is osseointegration?
- How does smoking affect implants?

Respond with: DOMAIN

Your answer must be exactly one word: SQL or DOMAIN
""".strip()

    try:
        response = llm.invoke(routing_prompt)
        if isinstance(response, str):
            response_text = response.strip().upper()
        elif isinstance(response, dict):
            response_text = response.get("result", "").strip().upper()
        else:
            response_text = str(response).strip().upper()

        # Sanitize extra characters
        response_text = response_text.split()[0].replace(".", "")

        print("🧭 Routing Response:", repr(response_text))  # Debug print
        return response_text == "SQL"

    except Exception as e:
        print("⚠️ Routing failed:", e)
        return False

In [178]:
def is_sql_question(question, LLM):
    routing_prompt = f"""
You are a smart assistant helping users with dental implant questions.

You can access a table with the following columns:
- MANUFACTURER_D_NAME (manufacturer name)
- cause (categorical reason for failure)
- year (report year)

Valid values for `cause` include:
- PST: Poor surgical technique
- IPF: Inability to achieve primary fixation
- IIL: Inadvertent implant loading
- INF: Infection
- UCD: Uncontrolled diabetes
- BF: Biological failures
- MF: Mechanical failures
- OSS: Osseointegration failure
- NO CLEAR CAUSE: Unclear or unspecified

A user asks:
"{question}"

If the question requires querying counts, comparisons, or trends from the table — such as:
- What are the most common causes of implant failure?
- Which manufacturer had more infections?
- Show the yearly trend of implant failure.

Respond with only one word: SQL

If the question asks for general knowledge (e.g., definitions, effects) that cannot be answered from the table:
- What is osseointegration?
- How does smoking affect implants?

Respond with only one word: DOMAIN
""".strip()

    try:
        response = LLM.invoke(routing_prompt)
        if isinstance(response, str):
            response_text = response.strip().upper()
        elif isinstance(response, dict):
            response_text = response.get("result", "").strip().upper()
        else:
            response_text = str(response).strip().upper()

        # Sanitize extra characters
        response_text = response_text.split()[0].replace(".", "")

        print("🧭 Routing Response:", repr(response_text))  # Debug print
        return response_text == "SQL"

    except Exception as e:
        print("⚠️ Routing failed:", e)
        return False

In [179]:
question = "What is the influence of infection on dental implant failure?"

In [180]:
question = "Which manufacturer had more mechanical failures?"

In [186]:
def is_sql_question(question, LLM):
    routing_prompt = f"""
You are a smart assistant helping users with dental implant questions.

You can access a table with the following columns:
- MANUFACTURER_D_NAME (manufacturer name)
- cause (categorical reason for failure)
- year (report year)

Valid values for `cause` include:
- PST: Poor surgical technique
- IPF: Inability to achieve primary fixation
- IIL: Inadvertent implant loading
- INF: Infection
- UCD: Uncontrolled diabetes
- BF: Biological failures
- MF: Mechanical failures
- OSS: Osseointegration failure
- NO CLEAR CAUSE: Unclear or unspecified

A user asks:
"{question}"

If the question requires querying counts, comparisons, or trends from the table — such as:
- What are the most common causes of implant failure?
- Which manufacturer had more infections?
- Show the yearly trend of implant failure.

Respond with: SQL

If the question asks for general knowledge (e.g., definitions, effects) that cannot be answered from the table:
- What is osseointegration?
- How does smoking affect implants?

Respond with: DOMAIN

Your answer must be exactly one word: SQL or DOMAIN
""".strip()

    try:
        response = LLM.invoke(routing_prompt)
        if isinstance(response, str):
            response_text = response.strip()
        elif isinstance(response, dict):
            response_text = response.get("result", "").strip()
        else:
            response_text = str(response).strip()

        # ✅ Robust match for SQL or DOMAIN
        import re
        match = re.search(r"\b(SQL|DOMAIN)\b", response_text.upper())
        if match:
            response_text = match.group(1)
        else:
            print("⚠️ Could not match SQL or DOMAIN in:", response_text)
            return False

        print("🧭 Routing Response:", repr(response_text))
        return response_text == "SQL"

    except Exception as e:
        print("⚠️ Routing failed:", e)
        return False

In [187]:
def get_clean_sql(question, llm):
    prompt = generate_sql_prompt(question)

    # ✅ Correctly extract content from AIMessage
    response_obj = llm.invoke(prompt)
    raw_response = response_obj.content.strip()

    # 🧹 Clean and extract SQL block
    if "```sql" in raw_response:
        sql_code = raw_response.split("```sql")[1].split("```")[0].strip()
    else:
        sql_code = raw_response.strip()

    if ";" in sql_code:
        sql_code = sql_code.split(";")[0].strip()

    return sql_code

In [188]:
if is_sql_question(question, llm_online):
    sql = get_clean_sql(question, llm_online)
    result_df = run_sql(sql, {"labeled_dental_reports": df_sample})
    answer = generate_answer_from_result(question, result_df, llm=llm_online)
else:
    print('...')
    answer = rag_chain_online.invoke(question)["result"].strip()
print("\n✅ Final Answer:\n", answer)

🧭 Routing Response: 'SQL'

✅ Final Answer:
                     MANUFACTURER_D_NAME  mechanical_failures
                          ZIMMER DENTAL                  113
                              BIOMET 3I                   47
IMPLANT DIRECT SYBRON MANUFACTURING LLC                   14
                         BIOTECH DENTAL                    4
                       BIOHORIZONS INC.                    2
                               NEOSS AB                    1
                          HIOSSEN, INC.                    1
                 HAGER & MEISINGER GMBH                    1
            BIOHORIZONS IMPLANT SYSTEMS                    1

The manufacturer with the highest number of mechanical failure cases is ZIMMER DENTAL with 113 reports.


In [189]:
import pandas as pd

# Benchmark list
benchmark = [
    {
        "question": "What are the most common causes of implant failure in the data?",
        "expected_sql": "SELECT cause, COUNT(*) as count FROM labeled_dental_reports GROUP BY cause ORDER BY count DESC;"
    },
    {
        "question": "Is the number of implant failures increasing over time?",
        "expected_sql": "SELECT year, COUNT(*) as total_failures FROM labeled_dental_reports GROUP BY year ORDER BY year;"
    },
    {
        "question": "Which manufacturer had the highest number of mechanical failures?",
        "expected_sql": "SELECT MANUFACTURER_D_NAME, COUNT(*) as mf_count FROM labeled_dental_reports WHERE cause = 'MF' GROUP BY MANUFACTURER_D_NAME ORDER BY mf_count DESC;"
    },
    {
        "question": "How many cases were reported in 2023?",
        "expected_sql": "SELECT COUNT(*) as case_count FROM labeled_dental_reports WHERE year = 2023;"
    },
    {
        "question": "Show the yearly trend of infection cases.",
        "expected_sql": "SELECT year, COUNT(*) as infection_cases FROM labeled_dental_reports WHERE cause = 'INF' GROUP BY year ORDER BY year;"
    },
    {
        "question": "What are the top 3 manufacturers with the most total cases?",
        "expected_sql": "SELECT MANUFACTURER_D_NAME, COUNT(*) as total_cases FROM labeled_dental_reports GROUP BY MANUFACTURER_D_NAME ORDER BY total_cases DESC LIMIT 3;"
    },
    {
        "question": "What is the least reported cause?",
        "expected_sql": "SELECT cause, COUNT(*) as count FROM labeled_dental_reports GROUP BY cause ORDER BY count ASC LIMIT 1;"
    }
]
# Evaluate
sql_eval_results = []

for item in benchmark:
    question = item["question"]
    gold_sql = item["expected_sql"]
    
    try:
        if is_sql_question(question, llm):
            pred_sql = get_clean_sql(question, llm)
        else:
            pred_sql = "N/A"
    except Exception as e:
        pred_sql = f"ERROR: {str(e)}"

    sql_eval_results.append({
        "question": question,
        "gold": gold_sql,
        "pred": pred_sql
    })

# Show results in DataFrame
df_eval = pd.DataFrame(sql_eval_results)
df_eval

🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'


,question,gold,pred
0,What are the most common causes of implant fai...,"SELECT cause, COUNT(*) as count FROM labeled_d...",ERROR: 'str' object has no attribute 'content'
1,Is the number of implant failures increasing o...,"SELECT year, COUNT(*) as total_failures FROM l...",ERROR: 'str' object has no attribute 'content'
2,Which manufacturer had the highest number of m...,"SELECT MANUFACTURER_D_NAME, COUNT(*) as mf_cou...",ERROR: 'str' object has no attribute 'content'
3,How many cases were reported in 2023?,SELECT COUNT(*) as case_count FROM labeled_den...,ERROR: 'str' object has no attribute 'content'
4,Show the yearly trend of infection cases.,"SELECT year, COUNT(*) as infection_cases FROM ...",ERROR: 'str' object has no attribute 'content'
5,What are the top 3 manufacturers with the most...,"SELECT MANUFACTURER_D_NAME, COUNT(*) as total_...",ERROR: 'str' object has no attribute 'content'
6,What is the least reported cause?,"SELECT cause, COUNT(*) as count FROM labeled_d...",ERROR: 'str' object has no attribute 'content'


In [73]:
import pandas as pd
import json

# Load benchmark from JSON
with open("sql_eval_benchmark_50.json", "r") as f:
    benchmark = json.load(f)

# Evaluation loop
sql_eval_results = []

for item in benchmark:
    question = item["question"]
    gold_sql = item["expected_sql"]

    try:
        if is_sql_question(question, llm):
            pred_sql = get_clean_sql(question, llm)
        else:
            pred_sql = "N/A"
    except Exception as e:
        pred_sql = f"ERROR: {str(e)}"

    sql_eval_results.append({
        "question": question,
        "gold": gold_sql,
        "pred": pred_sql
    })

# Convert to DataFrame
df_eval = pd.DataFrame(sql_eval_results)
df_eval

🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Re

,question,gold,pred
0,What are the most common causes of implant fai...,"SELECT cause, COUNT(*) as count FROM labeled_d...","SELECT \n cause, \n COUNT(*) as count\nF..."
1,Is the number of implant failures increasing o...,"SELECT year, COUNT(*) as total_failures FROM l...","SELECT \n year,\n COUNT(*) as failures\n..."
2,Which manufacturer had the highest number of m...,"SELECT MANUFACTURER_D_NAME, COUNT(*) as mf_cou...","SELECT \n MANUFACTURER_D_NAME,\n COUNT(*..."
3,How many cases were reported in 2023?,SELECT COUNT(*) as case_count FROM labeled_den...,SELECT \n COUNT(*) AS case_count\nFROM labe...
4,Show the yearly trend of infection cases.,"SELECT year, COUNT(*) as infection_cases FROM ...","SELECT year, COUNT(*) AS infection_cases FROM ..."
5,What are the top 3 manufacturers with the most...,"SELECT MANUFACTURER_D_NAME, COUNT(*) as total_...","SELECT \n MANUFACTURER_D_NAME, \n SUM(CA..."
6,What is the least reported cause?,"SELECT cause, COUNT(*) as count FROM labeled_d...","SELECT\n cause,\n COUNT(*) as count\nFRO..."
7,Which manufacturer had the highest number of c...,"SELECT MANUFACTURER_D_NAME, COUNT(*) as count ...","SELECT\n MANUFACTURER_D_NAME,\n COUNT(*)..."
8,How many osseointegration failures occurred ea...,"SELECT year, COUNT(*) as oss_failures FROM lab...","SELECT \n year, \n COUNT(CASE WHEN cause..."
9,Which year had the highest number of biologica...,"SELECT year, COUNT(*) as count FROM labeled_de...","SELECT \n year, \n COUNT(*) as count\nFR..."


In [159]:
import pandas as pd
import json

# Load benchmark from JSON
with open("sql_eval_benchmark_50.json", "r") as f:
    benchmark = json.load(f)

# Evaluation loop
sql_eval_results = []

for item in benchmark:
    question = item["question"]
    gold_sql = item["expected_sql"]

    try:
        if is_sql_question(question, llm_online):
            pred_sql = get_clean_sql(question, llm_online)
        else:
            pred_sql = "N/A"
    except Exception as e:
        pred_sql = f"ERROR: {str(e)}"

    sql_eval_results.append({
        "question": question,
        "gold": gold_sql,
        "pred": pred_sql
    })

# Convert to DataFrame
df_eval = pd.DataFrame(sql_eval_results)
df_eval

🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Response: 'SQL'
🧭 Routing Re

,question,gold,pred
0,What are the most common causes of implant fai...,"SELECT cause, COUNT(*) as count FROM labeled_d...","SELECT\n cause,\n COUNT(*) as count\nFRO..."
1,Is the number of implant failures increasing o...,"SELECT year, COUNT(*) as total_failures FROM l...","SELECT\n year,\n COUNT(*) as implant_fai..."
2,Which manufacturer had the highest number of m...,"SELECT MANUFACTURER_D_NAME, COUNT(*) as mf_cou...","SELECT\n MANUFACTURER_D_NAME,\n COUNT(*)..."
3,How many cases were reported in 2023?,SELECT COUNT(*) as case_count FROM labeled_den...,SELECT\n COUNT(*) as case_count\nFROM label...
4,Show the yearly trend of infection cases.,"SELECT year, COUNT(*) as infection_cases FROM ...","SELECT\n year,\n COUNT(*) as infection_c..."
5,What are the top 3 manufacturers with the most...,"SELECT MANUFACTURER_D_NAME, COUNT(*) as total_...",Generated SQL:\nSELECT\n MANUFACTURER_D_NAM...
6,What is the least reported cause?,"SELECT cause, COUNT(*) as count FROM labeled_d...","SELECT\n cause,\n COUNT(*) as count\nFRO..."
7,Which manufacturer had the highest number of c...,"SELECT MANUFACTURER_D_NAME, COUNT(*) as count ...","SELECT\n MANUFACTURER_D_NAME,\n COUNT(*)..."
8,How many osseointegration failures occurred ea...,"SELECT year, COUNT(*) as oss_failures FROM lab...","SELECT\n year,\n COUNT(*) as osseointegr..."
9,Which year had the highest number of biologica...,"SELECT year, COUNT(*) as count FROM labeled_de...","SELECT\n year,\n COUNT(*) as biological_..."


In [160]:
import difflib

# Helper to normalize SQL strings
def normalize_sql(sql):
    return (
        sql.strip()
        .lower()
        .replace("\n", " ")
        .replace(";", "")
        .replace("  ", " ")
    )

# Fuzzy similarity score between gold and predicted SQL
def fuzzy_score(row):
    gold = normalize_sql(row["gold"])
    pred = normalize_sql(row["pred"])
    return difflib.SequenceMatcher(None, gold, pred).ratio()

# Apply fuzzy score evaluation
df_eval["fuzzy_score"] = df_eval.apply(fuzzy_score, axis=1)

# Show summary
print(f"✅ Average Fuzzy Match Score: {df_eval['fuzzy_score'].mean():.2%}")

# Optionally display result
df_eval[["question", "fuzzy_score", "gold", "pred"]]

✅ Average Fuzzy Match Score: 89.35%


,question,fuzzy_score,gold,pred
0,What are the most common causes of implant fai...,0.899522,"SELECT cause, COUNT(*) as count FROM labeled_d...","SELECT\n cause,\n COUNT(*) as count\nFRO..."
1,Is the number of implant failures increasing o...,0.798246,"SELECT year, COUNT(*) as total_failures FROM l...","SELECT\n year,\n COUNT(*) as implant_fai..."
2,Which manufacturer had the highest number of m...,0.835366,"SELECT MANUFACTURER_D_NAME, COUNT(*) as mf_cou...","SELECT\n MANUFACTURER_D_NAME,\n COUNT(*)..."
3,How many cases were reported in 2023?,0.986842,SELECT COUNT(*) as case_count FROM labeled_den...,SELECT\n COUNT(*) as case_count\nFROM label...
4,Show the yearly trend of infection cases.,0.983051,"SELECT year, COUNT(*) as infection_cases FROM ...","SELECT\n year,\n COUNT(*) as infection_c..."
5,What are the top 3 manufacturers with the most...,0.937294,"SELECT MANUFACTURER_D_NAME, COUNT(*) as total_...",Generated SQL:\nSELECT\n MANUFACTURER_D_NAM...
6,What is the least reported cause?,0.960396,"SELECT cause, COUNT(*) as count FROM labeled_d...","SELECT\n cause,\n COUNT(*) as count\nFRO..."
7,Which manufacturer had the highest number of c...,0.958904,"SELECT MANUFACTURER_D_NAME, COUNT(*) as count ...","SELECT\n MANUFACTURER_D_NAME,\n COUNT(*)..."
8,How many osseointegration failures occurred ea...,0.930041,"SELECT year, COUNT(*) as oss_failures FROM lab...","SELECT\n year,\n COUNT(*) as osseointegr..."
9,Which year had the highest number of biologica...,0.809859,"SELECT year, COUNT(*) as count FROM labeled_de...","SELECT\n year,\n COUNT(*) as biological_..."


In [163]:
# Register df_sample to DuckDB
import duckdb
con = duckdb.connect(database=':memory:')
con.register('labeled_dental_reports', df_sample)  # use this alias to match SQL

# STEP 2: SQL runner with robust error handling
def run_sql(query):
    try:
        df = con.execute(query).fetchdf()
        if df is None or not isinstance(df, pd.DataFrame):
            print(f"❌ Query returned None or invalid DataFrame:\n{query}")
            return pd.DataFrame()
        return df
    except Exception as e:
        print(f"❌ SQL Error:\n{query}\nError: {e}")
        return pd.DataFrame()


In [164]:
i = 0  # Change this index to inspect a different question

print(f"🔍 Question: {df_eval.loc[i, 'question']}\n")

# Gold SQL
print("🟡 Gold SQL:")
print(df_eval.loc[i, 'gold'])
gold_df = run_sql(df_eval.loc[i, 'gold'])

# Predicted SQL
print("\n🔵 Predicted SQL:")
print(df_eval.loc[i, 'pred'])
pred_df = run_sql(df_eval.loc[i, 'pred'])

# Display results
print("\n✅ Gold SQL Result:")
display(gold_df)

print("\n🎯 Predicted SQL Result:")
display(pred_df)

🔍 Question: What are the most common causes of implant failure in the data?

🟡 Gold SQL:
SELECT cause, COUNT(*) as count FROM labeled_dental_reports GROUP BY cause ORDER BY count DESC;

🔵 Predicted SQL:
SELECT
    cause,
    COUNT(*) as count
FROM labeled_dental_reports
GROUP BY cause
ORDER BY count DESC
LIMIT 1 OFFSET 1

✅ Gold SQL Result:


,cause,count
0,OSS,4259
1,NO CLEAR CAUSE,3332
2,PST,871
3,INF,511
4,BF,420
5,IPF,348
6,MF,184
7,IIL,40
8,UCD,34



🎯 Predicted SQL Result:


,cause,count
0,NO CLEAR CAUSE,3332


In [165]:
import re

def rows_to_set(df):
    return set(tuple(row) for row in df.to_numpy())

def extract_limit(sql):
    """Extract LIMIT value from SQL if present."""
    match = re.search(r"LIMIT\s+(\d+)", sql, re.IGNORECASE)
    return int(match.group(1)) if match else None

def normalize_columns(df):
    """Rename all columns to generic names (col1, col2, ...) for fair comparison."""
    df = df.copy()
    df.columns = [f"col{i+1}" for i in range(df.shape[1])]
    return df

execution_match_scores = []

for i in range(len(df_eval)):
    gold_sql = df_eval.loc[i, "gold"]
    pred_sql = df_eval.loc[i, "pred"]

    gold_df = run_sql(gold_sql)
    pred_df = run_sql(pred_sql)

    if gold_df.empty or pred_df.empty:
        execution_match_scores.append(0.0)
        print(f"❌ Row {i}: Empty result (gold or pred)")
        continue

    # Normalize columns for semantic equality
    gold_df = normalize_columns(gold_df)
    pred_df = normalize_columns(pred_df)

    gold_limit = extract_limit(gold_sql)
    pred_limit = extract_limit(pred_sql)

    # Determine top-k comparison size
    if gold_limit or pred_limit:
        k = min(
            gold_limit if gold_limit else float('inf'),
            pred_limit if pred_limit else float('inf')
        )
        if len(gold_df) < k or len(pred_df) < k:
            execution_match_scores.append(0.0)
            print(f"❌ Row {i}: Not enough rows for top-{k} comparison")
            continue

        gold_topk = gold_df.head(k).reset_index(drop=True)
        pred_topk = pred_df.head(k).reset_index(drop=True)

        if gold_topk.equals(pred_topk):
            execution_match_scores.append(1.0)
            print(f"✅ Row {i}: Top-{k} rows match")
        else:
            execution_match_scores.append(0.0)
            print(f"❌ Row {i}: Top-{k} rows mismatch")
    else:
        # Fallback to unordered set comparison
        gold_set = rows_to_set(gold_df)
        pred_set = rows_to_set(pred_df)

        if gold_set == pred_set:
            execution_match_scores.append(1.0)
            print(f"✅ Row {i}: Set match")
        else:
            execution_match_scores.append(0.0)
            print(f"❌ Row {i}: Set mismatch")

# Update the result_match column
df_eval["result_match"] = execution_match_scores

# Final score
accuracy = df_eval["result_match"].mean()
print(f"\n📊 Final Execution Match Accuracy: {accuracy:.1%}")

❌ Row 0: Top-1 rows mismatch
❌ Row 1: Empty result (gold or pred)
✅ Row 2: Top-1 rows match
✅ Row 3: Set match
✅ Row 4: Set match
❌ SQL Error:
Generated SQL:
SELECT
    MANUFACTURER_D_NAME,
    COUNT(*) as total_cases
FROM labeled_dental_reports
GROUP BY MANUFACTURER_D_NAME
ORDER BY total_cases DESC
LIMIT 3
Error: Parser Error: syntax error at or near "Generated"
❌ Row 5: Empty result (gold or pred)
✅ Row 6: Top-1 rows match
✅ Row 7: Top-1 rows match
✅ Row 8: Set match
✅ Row 9: Top-1 rows match
✅ Row 10: Set match
✅ Row 11: Top-1 rows match
✅ Row 12: Set match
✅ Row 13: Set match
✅ Row 14: Top-1 rows match
✅ Row 15: Set match
❌ Row 16: Set mismatch
✅ Row 17: Set match
✅ Row 18: Top-1 rows match
✅ Row 19: Top-1 rows match
❌ Row 20: Top-1 rows mismatch
❌ Row 21: Set mismatch
✅ Row 22: Top-1 rows match
✅ Row 23: Set match
✅ Row 24: Set match
✅ Row 25: Top-1 rows match
✅ Row 26: Set match
✅ Row 27: Top-1 rows match
✅ Row 28: Set match
✅ Row 29: Top-5 rows match
✅ Row 30: Top-1 rows match
✅

In [132]:
df_sample.head()

,MDR_REPORT_KEY,MANUFACTURER_D_NAME,FOI_TEXT,year,cause
0,9585508,IMPLANT DIRECT SYBRON MANUFACTURING LLC,"PER COMPLAINT (B)(4), DURING CLINICAL PROCEDUR...",2020,OSS
1,9817202,IMPLANT DIRECT SYBRON MANUFACTURING LLC,FOLLOW-UP SUBMITTED TO REPORT DEVICE EVALUATIO...,2020,NO CLEAR CAUSE
2,9809640,IMPLANT DIRECT SYBRON MANUFACTURING LLC,FOLLOW-UP SUBMITTED TO REPORT DEVICE EVALUATIO...,2020,NO CLEAR CAUSE
3,9971673,IMPLANT DIRECT SYBRON MANUFACTURING LLC,INCLUDED NI FOR SECTION TO INDICATE NO INFORMA...,2020,NO CLEAR CAUSE
4,10521532,ZIMMER DENTAL,THIS REPORT IS BEING SUBMITTED TO RELAY ADDITI...,2020,NO CLEAR CAUSE


In [190]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Load benchmark
benchmark = [
  {
    "question": "What are the most common causes of implant failure?",
    "expected_sql": "SELECT cause, COUNT(*) as count FROM labeled_dental_reports GROUP BY cause ORDER BY count DESC;",
    "expected_answer": "The most common causes of implant failure are ordered by frequency in the result, with the top one being the most frequent."
  },
  {
    "question": "Which manufacturer had more infections: A or B?",
    "expected_sql": "SELECT MANUFACTURER_D_NAME, COUNT(*) as infection_count FROM labeled_dental_reports WHERE cause = 'INFECTION' AND MANUFACTURER_D_NAME IN ('A', 'B') GROUP BY MANUFACTURER_D_NAME ORDER BY infection_count DESC;",
    "expected_answer": "The manufacturer with more infection-related cases between A and B is listed first."
  },
  {
    "question": "Is the number of implant failures increasing over time?",
    "expected_sql": "SELECT year_x, COUNT(*) as total_failures FROM labeled_dental_reports GROUP BY year_x ORDER BY year_x;",
    "expected_answer": "The trend of implant failures over time is shown. You can assess whether it's increasing by observing the counts."
  },
  {
    "question": "Has pain-related failure decreased in recent years?",
    "expected_sql": "SELECT year_x, COUNT(*) as pain_failures FROM labeled_dental_reports WHERE cause = 'PAIN' GROUP BY year_x ORDER BY year_x;",
    "expected_answer": "The trend of pain-related failures over time is shown. Decrease can be assessed from the result."
  },
  {
    "question": "Which manufacturer had the highest number of inflammation cases?",
    "expected_sql": "SELECT MANUFACTURER_D_NAME, COUNT(*) as inflammation_cases FROM labeled_dental_reports WHERE cause = 'INFLAMMATION' GROUP BY MANUFACTURER_D_NAME ORDER BY inflammation_cases DESC;",
    "expected_answer": "The manufacturer with the highest number of inflammation-related cases is listed at the top."
  },
  {
    "question": "How many cases were reported in 2023?",
    "expected_sql": "SELECT COUNT(*) as case_count FROM labeled_dental_reports WHERE year_x = 2023;",
    "expected_answer": "The number of cases reported in 2023 is returned as a single count."
  },
  {
    "question": "How many infection cases did manufacturer A have in 2022?",
    "expected_sql": "SELECT COUNT(*) as infection_count FROM labeled_dental_reports WHERE MANUFACTURER_D_NAME = 'A' AND cause = 'INFECTION' AND year_x = 2022;",
    "expected_answer": "The number of infection cases reported for manufacturer A in 2022."
  },
  {
    "question": "Show the yearly trend of inflammation cases.",
    "expected_sql": "SELECT year_x, COUNT(*) as inflammation_cases FROM labeled_dental_reports WHERE cause = 'INFLAMMATION' GROUP BY year_x ORDER BY year_x;",
    "expected_answer": "The trend of inflammation cases over the years."
  },
  {
    "question": "What are the top 3 manufacturers with the most total cases?",
    "expected_sql": "SELECT MANUFACTURER_D_NAME, COUNT(*) as total_cases FROM labeled_dental_reports GROUP BY MANUFACTURER_D_NAME ORDER BY total_cases DESC LIMIT 3;",
    "expected_answer": "The top 3 manufacturers with the highest number of adverse event reports."
  },
  {
    "question": "What is the least reported cause?",
    "expected_sql": "SELECT cause, COUNT(*) as count FROM labeled_dental_reports GROUP BY cause ORDER BY count ASC LIMIT 1;",
    "expected_answer": "The cause with the lowest number of reports is listed."
  }
    # Add more test cases here
]

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def eval_answer_similarity(a1, a2):
    e1 = embedder.encode(a1, convert_to_tensor=True)
    e2 = embedder.encode(a2, convert_to_tensor=True)
    return float(util.cos_sim(e1, e2))

results = []

for row in benchmark:
    question = row["question"]
    expected_sql = row["expected_sql"]
    expected_answer = row["expected_answer"]

    if is_sql_question(question, llm):
        generated_sql = get_clean_sql(question, llm)
        result_df = run_sql(generated_sql, {"labeled_dental_reports": df_sample})
        generated_answer = generate_answer_from_result(question, result_df, llm=llm)
    else:
        generated_sql = "N/A"
        generated_answer = rag_chain.invoke(question)["result"].strip()

    sql_match = (expected_sql.strip().lower() == generated_sql.strip().lower())
    answer_sim = eval_answer_similarity(expected_answer, generated_answer)

    results.append({
        "question": question,
        "expected_sql": expected_sql,
        "generated_sql": generated_sql,
        "sql_match": sql_match,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "answer_similarity": round(answer_sim, 4)
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

🧭 Routing Response: 'SQL'


AttributeError: 'str' object has no attribute 'content'

In [83]:
from langchain.memory import ConversationBufferMemory 
from langchain.chains import ConversationalRetrievalChain

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

rag_conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,  # your OllamaLLM or other LLM
    retriever=vectorstore.as_retriever(),
    memory=memory,
    return_source_documents=True
)

/var/folders/_6/bkb79v1n1qgb5pjgnzbz79nw0000gn/T/ipykernel_19209/1946944477.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [119]:
import tkinter as tk

# llm = OllamaLLM(model="llama3", temperature=0.2)

chat_history = []  # global conversation state

def launch_ui(llm):
    def interpret_and_query():
        user_q = query_entry.get()
        result_output.insert(tk.END, f"\n\n🧑‍💻 You: {user_q}\n")

        if is_sql_question(user_q, llm):
            sql = get_clean_sql(user_q, llm)
            result_output.insert(tk.END, f"\n📝 Generated SQL:\n{sql}\n")

            result_df = run_sql(sql, {"labeled_dental_reports": df_sample})
            answer = generate_answer_from_result(user_q, result_df, llm=llm)
        else:
            full_context = "\n".join([f"User: {q}\nAssistant: {a}" for q, a in chat_history])
            prompt = f"""{full_context}\nUser: {user_q}\nAssistant:"""
            answer = llm.invoke(prompt).strip()
            chat_history.append((user_q, answer))

        result_output.insert(tk.END, f"\n🤖 Answer:\n{answer}\n")
        query_entry.delete(0, tk.END)

    # UI Setup
    root = tk.Tk()
    root.title("Dental Device Conversational Assistant")

    tk.Label(root, text="Ask a question:").pack()
    query_entry = tk.Entry(root, width=100)
    query_entry.pack()

    tk.Button(root, text="Submit", command=interpret_and_query).pack()

    result_output = tk.Text(root, height=25, width=100, wrap="word")
    result_output.pack()

    root.mainloop()

# Run the UI
launch_ui(llm)

⚠️ Routing failed: model 'llama3:8b' not found (status code: 404)


Exception in Tkinter callback
Traceback (most recent call last):
  File "/opt/anaconda3/envs/NLP/lib/python3.11/tkinter/__init__.py", line 1967, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "/var/folders/_6/bkb79v1n1qgb5pjgnzbz79nw0000gn/T/ipykernel_57701/946702503.py", line 21, in interpret_and_query
    answer = llm.invoke(prompt).strip()
             ^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/NLP/lib/python3.11/site-packages/langchain_core/language_models/llms.py", line 389, in invoke
    self.generate_prompt(
  File "/opt/anaconda3/envs/NLP/lib/python3.11/site-packages/langchain_core/language_models/llms.py", line 766, in generate_prompt
    return self.generate(prompt_strings, stop=stop, callbacks=callbacks, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/NLP/lib/python3.11/site-packages/langchain_core/language_models/llms.py", line 973, in generate
    return self._generate_

In [84]:
question = "Which manufacturer has the most INFLAMMATION cases?"

# Step 1: Use a general-purpose LLM (e.g., OllamaLLM) for SQL generation
sql = get_clean_sql(question, rag_chain)
print("Generated SQL:\n", sql)

# Step 2: Run the SQL query against your pandas DataFrame
result_df = run_sql(sql, {"labeled_dental_reports": df_sample})

# Step 3: Use RAG chain to generate a domain-informed human answer
answer = generate_answer_from_result(question, result_df, llm=rag_chain)

print("\n✅ Final Answer:\n", answer)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-...AfUA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [191]:
question = "Which manufacturer has the most INFLAMMATION cases?"

sql = get_clean_sql(question, llm)
print("Generated SQL:\n", sql)

result_df = run_sql(sql, {"labeled_dental_reports": df_sample})

answer = generate_answer_from_result(question, result_df)
print("\n✅ Final Answer:\n", answer)

AttributeError: 'str' object has no attribute 'content'

In [192]:
import tkinter as tk

llm = OllamaLLM(model="llama3:8b", temperature=0.2)
def launch_ui(llm):
    def interpret_and_query():
        user_q = query_entry.get()
        sql = get_clean_sql(user_q, llm)
        result_output.delete("1.0", tk.END)
        result_output.insert(tk.END, f"Generated SQL:\n{sql}\n\n")

        result_df = run_sql(sql, {"labeled_dental_reports": df_sample})
        answer = generate_answer_from_result(user_q, result_df, llm)

        result_output.insert(tk.END, f"Answer:\n{answer}")

    root = tk.Tk()
    root.title("Dental Device SQL Assistant")

    tk.Label(root, text="Ask a question:").pack()
    query_entry = tk.Entry(root, width=100)
    query_entry.pack()

    tk.Button(root, text="Submit", command=interpret_and_query).pack()

    result_output = tk.Text(root, height=20, width=100, wrap="word")
    result_output.pack()

    root.mainloop()

# Run the UI
launch_ui(llm)